In [1]:
import pandas as pd 
import os 

results_path = "../Results/Sunnybrook"

df = pd.DataFrame()
models = sorted(os.listdir(results_path))
for folder in models:
    if os.path.isdir(os.path.join(results_path,folder)):
        df_temp = pd.read_csv(os.path.join(results_path,folder,"results.csv"))
        df_temp["Model"] = "HybridGNet Dual" if "dual" in folder.lower() else "HybridGNet"
        # 2 if "2_10" in folder elif 3 if "3_10" in folder else 4
        df_temp["Resolutions"] = 2 if "2" in folder else 3 if "3" in folder else 4
        df = pd.concat([df,df_temp])
        
# read nnUNet results and append to df
nnunet_results = pd.read_csv("/home/ngaggion/Documents/Mask2Graph/Results/Sunnybrook/nnUNet_results.csv")
nnunet_results["Model"] = "nnUNet"
nnunet_results["Resolutions"] = 3  # assuming nnUNet is trained at 3mm resolution
df = pd.concat([df, nnunet_results])

# After creating the boxplots, add this code to generate the tables
for organ in df['organ'].unique():
    print(f"\n=== Table for {organ} ===")
    subdf = df[df['organ'] == organ]
    
    resolutions = subdf['Resolutions'].unique()
    models = subdf['Model'].unique()
    # Metrics
    metrics = ['DC', 'HD', 'ASSD']
    
    # Create MultiIndex columns
    columns = pd.MultiIndex.from_product([models, metrics])
    
    # Initialize DataFrame
    result_table = pd.DataFrame(index=resolutions, columns=columns)
    result_table.index.name = "Trained on"
    
    # Fill the table
    for resolution in resolutions:
        for model in models:
            filtered_data = subdf[(subdf['Resolutions'] == resolution) & (subdf['Model'] == model)]
            
            if not filtered_data.empty:
                result_table.loc[resolution, (model, 'DC')] = filtered_data['dc'].mean()
                result_table.loc[resolution, (model, 'HD')] = filtered_data['hd'].mean()
                result_table.loc[resolution, (model, 'ASSD')] = filtered_data['assd'].mean()
    
    # Format the numbers
    result_table = result_table.round(3)
    
    print(result_table)
    
# incorporate wilcoxon test to compare different models at different resolutions

from scipy.stats import wilcoxon
import numpy as np

for model in df["Model"].unique():
    subdf = df[df['Model'] == model]
    resolutions = subdf['Resolutions'].unique()

    for i in range(0, (len(resolutions))):
        for j in range(i + 1, len(resolutions)):
            res1 = resolutions[i]
            res2 = resolutions[j]

            data1 = subdf[subdf['Resolutions'] == res1]['dc']
            data2 = subdf[subdf['Resolutions'] == res2]['dc']

            stat, p_value = wilcoxon(data1, data2)
            # i need to say which is better 
            if p_value < 0.05:
                if np.mean(data1) > np.mean(data2):
                    print(f"{model} {res1} is better than {res2} with p-value: {p_value}")
                else:
                    print(f"{model} {res2} is better than {res1} with p-value: {p_value}")
            else:
                print(f"No significant difference between {model} {res1} and {res2} with p-value: {p_value}")


=== Table for LV endo ===
           HybridGNet                     HybridGNet Dual                      \
                   DC        HD      ASSD              DC        HD      ASSD   
Trained on                                                                      
2            0.882333  4.301712  1.690289        0.904019  3.653695  1.395536   
3            0.893398  4.013855  1.535036        0.902978  3.743602  1.416501   
4            0.900017  3.664317   1.41598        0.887177  3.886648  1.601237   

              nnUNet                      
                  DC        HD      ASSD  
Trained on                                
2                NaN       NaN       NaN  
3           0.918678  3.559276  1.186154  
4                NaN       NaN       NaN  

=== Table for LV epi ===
           HybridGNet                     HybridGNet Dual                      \
                   DC        HD      ASSD              DC        HD      ASSD   
Trained on                              

In [2]:
# Your original code that creates the 'results' table
def format_mean_std(group):
    return f"{group.mean():.3f} ± {group.std():.3f}"

# only keep models with 3 resolution
df = df[df['Resolutions'] == 3]

results = df.groupby(['organ', 'Model']).agg({
    'dc': format_mean_std,
    'hd': format_mean_std,
    'assd': format_mean_std
})
results.columns = ['DSC', 'HD', 'ASSD']

# Reshape: Organs as columns, Models as rows
reshaped = results.unstack(level=0)  # Move 'organ' from index to columns

# Reorder columns to group metrics by organ
organs = ['LV endo', 'LV epi']  # Adjust order as needed
metrics = ['DSC', 'HD', 'ASSD']

# Create new column order
new_columns = []
for organ in organs:
    for metric in metrics:
        new_columns.append((metric, organ))

# Reorder and display
reshaped_ordered = reshaped[new_columns]

print("=== Reshaped Table: Organs as Columns ===")
print(reshaped_ordered)

# Alternative: Flattened column names for cleaner display
flat_df = reshaped_ordered.copy()
flat_df.columns = [f"{organ} - {metric.replace(' Score', '').replace(' Distance', '')}" 
                   for metric, organ in flat_df.columns]

print("\n=== Cleaner Version with Flattened Column Names ===")
display(flat_df)

=== Reshaped Table: Organs as Columns ===
                           DSC             HD           ASSD            DSC  \
organ                  LV endo        LV endo        LV endo         LV epi   
Model                                                                         
HybridGNet       0.893 ± 0.099  4.014 ± 1.850  1.535 ± 0.760  0.655 ± 0.171   
HybridGNet Dual  0.903 ± 0.072  3.744 ± 1.792  1.417 ± 0.684  0.673 ± 0.180   
nnUNet           0.919 ± 0.056  3.559 ± 2.307  1.186 ± 0.550  0.734 ± 0.164   

                            HD           ASSD  
organ                   LV epi         LV epi  
Model                                          
HybridGNet       4.508 ± 2.332  1.385 ± 0.778  
HybridGNet Dual  4.553 ± 2.733  1.305 ± 0.694  
nnUNet           3.979 ± 2.552  0.997 ± 0.316  

=== Cleaner Version with Flattened Column Names ===


,LV endo - DSC,LV endo - HD,LV endo - ASSD,LV epi - DSC,LV epi - HD,LV epi - ASSD
Model,,,,,,
HybridGNet,0.893 ± 0.099,4.014 ± 1.850,1.535 ± 0.760,0.655 ± 0.171,4.508 ± 2.332,1.385 ± 0.778
HybridGNet Dual,0.903 ± 0.072,3.744 ± 1.792,1.417 ± 0.684,0.673 ± 0.180,4.553 ± 2.733,1.305 ± 0.694
nnUNet,0.919 ± 0.056,3.559 ± 2.307,1.186 ± 0.550,0.734 ± 0.164,3.979 ± 2.552,0.997 ± 0.316
